In [1]:
import sys, json, pathlib
from html import escape
from collections import defaultdict
from IPython.display import display, HTML

here = pathlib.Path().resolve()
root = here
for _ in range(4):
    if (root / 'grammar_errors').is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from grammar_errors import ErrorChecker

PARTITIONS = [
    root / 'processed_data' / 'sentences' / 'Student-2' / 'lesson-1' / 'paragraphs_custom.json',
    root / 'processed_data' / 'sentences' / 'Student-1' / 'lesson-3' / 'paragraphs_custom.json',
]

sessions = []
for p in PARTITIONS:
    data = json.loads(p.read_text(encoding='utf-8'))
    sessions.append(data)
    print(f"Loaded: {data['student']} / {data['lesson']} — {len(data['paragraphs'])} paragraphs")

Loaded: Student-2 / lesson-1 — 10 paragraphs
Loaded: Student-1 / lesson-3 — 10 paragraphs


In [2]:
# ── Run ErrorChecker on every session and paragraph ──────────────────────────
# errors_by_session[i][para_id] = list of error records
checker = ErrorChecker()

errors_by_session = []
for session in sessions:
    errors_by_para = {}
    print(f"\n── {session['student']} / {session['lesson']} ──")
    for para in session['paragraphs']:
        texts     = [s['text'] for s in para['sentences']]
        start_idx = para['sentences'][0]['index']
        records   = checker.check_sentences(texts, start_index=start_idx)
        errors_by_para[para['id']] = records
        print(f"  P{para['id']:02d} [{len(texts):3d} sent] {len(records):3d} errors — {para['label']}")
    errors_by_session.append(errors_by_para)

checker.close()
print("\nDone.")


── Student-2 / lesson-1 ──


  P01 [ 24 sent]   0 errors — Opening and excuse for being late


  P02 [ 26 sent]   6 errors — Weekend activities and Polish geography


  P03 [ 25 sent]  10 errors — Baltic Sea history and the Little Ice Age


  P04 [ 15 sent]   1 errors — Snow and weather in Warsaw


...

  P06 [ 26 sent]   1 errors — Scam calls and messaging apps across countries


  P07 [ 42 sent]   5 errors — Screen time, gaming, reading and digital habits


  P08 [ 19 sent]   3 errors — TikTok vs YouTube and algorithm attention


  P09 [ 42 sent]   6 errors — Discipline, idols and teenage years


  P10 [ 33 sent]   1 errors — Growing up and closing

Done.


In [3]:
# ── Helpers ───────────────────────────────────────────────────────────────────

DIM_COLORS = {
    'A': {'bg': '#DBEAFE', 'border': '#3B82F6', 'badge': '#1D4ED8', 'label': 'Sentence Architecture'},
    'B': {'bg': '#FEF3C7', 'border': '#F59E0B', 'badge': '#92400E', 'label': 'Tense & Aspect Mastery'},
    'C': {'bg': '#D1FAE5', 'border': '#10B981', 'badge': '#065F46', 'label': 'Nominal Precision'},
    'D': {'bg': '#EDE9FE', 'border': '#8B5CF6', 'badge': '#4C1D95', 'label': 'Modal & Functional Range'},
}

SEV_LABEL = {1: 'beginner', 2: 'medium', 3: 'high'}
SEV_ALPHA = {'beginner': '44', 'medium': '88', 'high': 'cc'}  # hex alpha for bg


def _word_count(text: str) -> int:
    return len(text.split())


ERROR_PENALTY_FACTOR = 5.0  # raise to harsher scoring, lower to soften

def compute_score(errors: list, sentences: list) -> int:
    """0-100 quality score: 100 * (1 - FACTOR * Swweights / words), clamped to 0."""
    total_words = sum(_word_count(s['text']) for s in sentences)
    if total_words == 0:
        return 100
    weighted_sum = sum(e['weight'] for e in errors)
    raw = ERROR_PENALTY_FACTOR * weighted_sum / total_words
    return max(0, round(100 * (1 - raw)))


def score_label(score: int):
    """Return (label, hex_color) for a score."""
    if score >= 80:
        return 'Good', '#16a34a'
    if score >= 60:
        return 'Moderate', '#d97706'
    if score >= 40:
        return 'Weak', '#ea580c'
    return 'Poor', '#dc2626'


def sentence_to_html(sentence: str, errors: list, uid_prefix: str) -> str:
    """Wrap error spans in a sentence with clickable tooltips."""
    errs = sorted(errors, key=lambda e: e['offset'])
    non_overlap, last_end = [], 0
    for e in errs:
        if e['offset'] >= last_end:
            non_overlap.append(e)
            last_end = e['offset'] + e['error_length']

    html, pos = '', 0
    for i, e in enumerate(non_overlap):
        html += escape(sentence[pos:e['offset']])
        dim   = e['dimension_code']
        col   = DIM_COLORS[dim]
        sev   = SEV_LABEL[e['weight']]
        alpha = SEV_ALPHA[sev]
        bg     = col['border'] + alpha
        border = col['border']
        uid    = uid_prefix + '_e' + str(i)

        suggestions = ', '.join(e['replacements'][:3]) if e['replacements'] else 'none'
        cat_esc     = escape(e['grammar_category'])
        dim_esc     = escape(e['dimension_label'])
        msg_esc     = escape(e['message'])
        sug_esc     = escape(suggestions)
        badge_col   = col['badge']
        wt          = e['weight']
        matched_esc = escape(sentence[e['offset']:e['offset'] + e['error_length']])

        tip_html = (
            "<b>" + cat_esc + "</b><br>"
            "<span style='color:" + badge_col + "'>" + dim_esc + "</span><br>"
            + msg_esc + "<br>"
            "<i>Suggestion:</i> " + sug_esc + "<br>"
            "<b>Severity:</b> " + sev + " &nbsp; <b>Weight:</b> " + str(wt) + "/5"
        )

        html += (
            '<span id="' + uid + '" class="err-span" '
            'style="background:' + bg + ';border-bottom:2px solid ' + border + ';'
            'border-radius:2px;padding:1px 2px;cursor:pointer;position:relative;" '
            'onclick="toggleTip(\'' + uid + '\')" '
            'title="' + cat_esc + ' | ' + sev + ' | weight ' + str(wt) + '">'
            + matched_esc +
            '<span id="tip_' + uid + '" class="err-tip" style="display:none;position:absolute;'
            'bottom:125%;left:0;z-index:999;background:white;border:1.5px solid ' + border + ';'
            'border-radius:6px;padding:8px 10px;min-width:220px;max-width:320px;'
            'box-shadow:0 4px 12px rgba(0,0,0,.15);font-size:12px;line-height:1.5;'
            'white-space:normal;text-align:left;">'
            + tip_html + '</span></span>'
        )
        pos = e['offset'] + e['error_length']

    html += escape(sentence[pos:])
    return html

print('Helpers loaded.')

Helpers loaded.


In [4]:
# ── render_report ─────────────────────────────────────────────────────────────

def render_report(partition: dict, errors_by_para: dict) -> None:
    js = """
    <script>
    function toggleTip(uid) {
        var tip = document.getElementById('tip_' + uid);
        if (!tip) return;
        var visible = tip.style.display !== 'none';
        document.querySelectorAll('.err-tip').forEach(function(t){ t.style.display='none'; });
        if (!visible) tip.style.display = 'block';
    }
    document.addEventListener('click', function(e){
        if (!e.target.closest('.err-span')) {
            document.querySelectorAll('.err-tip').forEach(function(t){ t.style.display='none'; });
        }
    });
    </script>
    """

    css = """
    <style>
    .para-card {
        font-family: Georgia, serif; font-size: 14px; line-height: 1.8;
        border: 1px solid #e2e8f0; border-radius: 8px; margin: 16px 0; overflow: hidden;
    }
    .para-header {
        background: #f8fafc; border-bottom: 1px solid #e2e8f0; padding: 8px 14px;
        font-family: sans-serif; font-size: 12px; color: #64748b;
        display: flex; justify-content: space-between; align-items: center;
    }
    .para-label { font-weight: 700; color: #1e293b; font-size: 13px; }
    .para-body  { padding: 12px 16px; }
    .para-footer {
        background: #f8fafc; border-top: 1px solid #e2e8f0; padding: 8px 14px;
        font-family: sans-serif; font-size: 12px; display: flex; align-items: center; gap: 16px;
    }
    .score-pill { font-weight: 800; font-size: 14px; padding: 2px 10px; border-radius: 12px; color: white; }
    .dim-legend { display: flex; gap: 10px; flex-wrap: wrap; margin-bottom: 10px; font-family: sans-serif; font-size: 11px; }
    .dim-chip   { padding: 2px 8px; border-radius: 10px; border: 1.5px solid; font-weight: 600; }
    </style>
    """

    legend = '<div class="dim-legend">'
    for code, c in DIM_COLORS.items():
        legend += ('<span class="dim-chip" style="background:' + c['bg'] + ';'
                   'border-color:' + c['border'] + ';color:' + c['badge'] + '">'
                   + code + ' \u2014 ' + c['label'] + '</span>')
    legend += '</div>'

    sev_legend = (
        '<div style="font-family:sans-serif;font-size:11px;color:#64748b;margin-bottom:14px;">'
        'Opacity = severity &nbsp;|&nbsp; '
        '<span style="opacity:.4">low (1\u20132)</span> &nbsp; '
        '<span style="opacity:.7">medium (3)</span> &nbsp; '
        '<span style="opacity:1">high (4\u20135)</span> &nbsp;|&nbsp; '
        'Click highlighted text for details.</div>'
    )

    student  = partition['student']
    lesson   = partition['lesson']
    body = js + css + ("<h3 style='font-family:sans-serif'>"
                       + student + " / " + lesson
                       + " \u2014 Paragraph Error Valoration</h3>"
                       + legend + sev_legend)

    for para in partition['paragraphs']:
        pid      = para['id']
        errors   = errors_by_para.get(pid, [])
        sentences = para['sentences']
        score    = compute_score(errors, sentences)
        slabel, scolor = score_label(score)

        errs_by_sent = defaultdict(list)
        for e in errors:
            errs_by_sent[e['sentence_index']].append(e)

        dim_counts = defaultdict(int)
        for e in errors:
            dim_counts[e['dimension_code']] += 1

        total_words = sum(_word_count(s['text']) for s in sentences)

        body += '<div class="para-card">'
        body += ('<div class="para-header">'
                 '<span class="para-label">P' + str(pid) + '. ' + escape(para['label']) + '</span>'
                 '<span>' + str(len(sentences)) + ' sentences &nbsp;\u00b7&nbsp; '
                 + str(total_words) + ' words &nbsp;\u00b7&nbsp; ' + str(len(errors)) + ' errors</span>'
                 '</div>')

        body += '<div class="para-body">'
        for s in sentences:
            sent_errors = errs_by_sent.get(s['index'], [])
            uid = "p" + str(pid) + "_s" + str(s['index'])
            body += sentence_to_html(s['text'], sent_errors, uid) + ' '
        body += '</div>'

        dim_chips = ''
        for code, cnt in sorted(dim_counts.items()):
            c = DIM_COLORS[code]
            dim_chips += ('<span style="background:' + c['bg'] + ';border:1px solid ' + c['border'] + ';'
                          'color:' + c['badge'] + ';padding:1px 7px;border-radius:9px;font-weight:600;">'
                          + code + ': ' + str(cnt) + '</span> ')

        body += ('<div class="para-footer">'
                 '<span class="score-pill" style="background:' + scolor + '">Score: ' + str(score) + '/100</span>'
                 '<span style="color:' + scolor + ';font-weight:600">' + slabel + '</span>'
                 '<span style="color:#94a3b8">|</span>'
                 + dim_chips + '</div>')
        body += '</div>'

    display(HTML(body))


# ── Run ───────────────────────────────────────────────────────────────────────
for session, errors_by_para in zip(sessions, errors_by_session):
    render_report(session, errors_by_para)

In [5]:
# ── render_summary ────────────────────────────────────────────────────────────

def render_summary(partition: dict, errors_by_para: dict) -> None:
    rows = ''
    for para in partition['paragraphs']:
        pid          = para['id']
        errors       = errors_by_para.get(pid, [])
        sentences    = para['sentences']
        score        = compute_score(errors, sentences)
        slabel, scolor = score_label(score)
        total_words  = sum(_word_count(s['text']) for s in sentences)
        weighted_sum = sum(e['weight'] for e in errors)

        bar = ('<div style="background:#e2e8f0;border-radius:4px;height:12px;width:120px;display:inline-block;">'
               '<div style="background:' + scolor + ';width:' + str(score) + '%;height:100%;border-radius:4px;"></div></div>')

        rows += ('<tr>'
                 '<td style="padding:6px 10px;font-weight:700;">P' + str(pid) + '</td>'
                 '<td style="padding:6px 10px;max-width:220px;">' + escape(para['label']) + '</td>'
                 '<td style="padding:6px 10px;text-align:center;">' + str(len(sentences)) + '</td>'
                 '<td style="padding:6px 10px;text-align:center;">' + str(total_words) + '</td>'
                 '<td style="padding:6px 10px;text-align:center;">' + str(len(errors)) + '</td>'
                 '<td style="padding:6px 10px;text-align:center;">' + str(weighted_sum) + '</td>'
                 '<td style="padding:6px 10px;">' + bar + ' &nbsp;<b style="color:' + scolor + '">' + str(score) + '</b></td>'
                 '<td style="padding:6px 10px;color:' + scolor + ';font-weight:600">' + slabel + '</td>'
                 '</tr>')

    html = ('<table style="border-collapse:collapse;font-family:sans-serif;font-size:13px;width:100%;">'
            '<thead><tr style="background:#f1f5f9;border-bottom:2px solid #cbd5e1;">'
            '<th style="padding:8px 10px;text-align:left">#</th>'
            '<th style="padding:8px 10px;text-align:left">Topic</th>'
            '<th style="padding:8px 10px">Sents</th>'
            '<th style="padding:8px 10px">Words</th>'
            '<th style="padding:8px 10px">Errors</th>'
            '<th style="padding:8px 10px">\u03a3 Weight</th>'
            '<th style="padding:8px 10px">Score</th>'
            '<th style="padding:8px 10px">Level</th>'
            '</tr></thead>'
            '<tbody>' + rows + '</tbody>'
            '</table>')
    display(HTML(html))


# ── Run ───────────────────────────────────────────────────────────────────────
for session, errors_by_para in zip(sessions, errors_by_session):
    display(HTML("<h3 style='font-family:sans-serif;margin-top:24px'>"
                 + session['student'] + " / " + session['lesson'] + " \u2014 Summary</h3>"))
    render_summary(session, errors_by_para)

#,Topic,Sents,Words,Errors,Σ Weight,Score,Level
P1,Opening and excuse for being late,24,127,0,0,100,Good
P2,Weekend activities and Polish geography,26,261,6,15,71,Moderate
P3,Baltic Sea history and the Little Ice Age,25,319,10,18,72,Moderate
P4,Snow and weather in Warsaw,15,130,1,1,96,Good
P5,"Trump, Ukraine and the Yalta comparison",40,552,9,22,80,Good
P6,Ukraine elections and Zelensky,35,404,7,16,80,Good
P7,Modern warfare and military equipment,45,512,8,15,85,Good
P8,NATO tactics and their limitations,50,473,4,12,87,Good
P9,North Korean troops and Trump–Poland relations,60,695,11,23,83,Good
P10,Polish presidential elections,48,551,6,13,88,Good


#,Topic,Sents,Words,Errors,Σ Weight,Score,Level
P1,Greeting and topic introduction,19,68,2,5,63,Moderate
P2,Study on notifications and the Stroop task,31,403,4,8,90,Good
P3,Personal experience turning off notifications,24,342,0,0,100,Good
P4,Benefits of no notifications and Twitter habits,28,353,3,6,92,Good
P5,Focus at work and the school exam disruption story,39,472,7,15,84,Good
P6,Scam calls and messaging apps across countries,26,339,1,3,96,Good
P7,"Screen time, gaming, reading and digital habits",42,474,5,11,88,Good
P8,TikTok vs YouTube and algorithm attention,19,178,3,7,80,Good
P9,"Discipline, idols and teenage years",42,339,6,14,79,Moderate
P10,Growing up and closing,33,385,1,2,97,Good
